# 📈 Pandas Part 8: Advanced Time Series Masterclass
**Time-Based Slicing, Resampling, Rolling Windows & Shifts**

Welcome to Part 8! Now that you can parse and clean dates (Part 7), it's time to manipulate time itself. In this notebook, we will master how to slice data using time strings, change data frequencies (Resampling), calculate moving averages (Rolling Windows), and measure change over time (Shifts & Differences).

Every concept follows our **16-Step Mastery Framework**:

`Theory → Mental Model → Diagram → Syntax → Example → Output → Explanation → Real World → Mistakes → Interview → Prediction → Practice (Levels 1-5)`


In [1]:
# ==========================================
# 🛠️ SETUP & INDUSTRY MOCK DATA
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print(f"Pandas Version: {pd.__version__}")

# 📊 Mock High-Frequency Retail Data (Hourly for 3 Months)
np.random.seed(42)
# Generate hourly timestamps from Jan 1 to Mar 31, 2023
dates = pd.date_range(start='2023-01-01', end='2023-03-31 23:00:00', freq='H')

# Simulate realistic hourly sales (higher during 10 AM - 8 PM, lower at night)
hour_factor = np.where((dates.hour >= 10) & (dates.hour <= 20), 1.5, 0.5)
base_sales = 100
noise = np.random.normal(0, 20, len(dates))
sales = (base_sales * hour_factor + noise).round(2)
sales = np.maximum(sales, 0) # Ensure no negative sales

# Create DataFrame
df_retail = pd.DataFrame({
    'Sales': sales,
    'Customers': (sales / 15).astype(int) + np.random.randint(1, 5, len(dates))
}, index=dates)

# Give the index a name
df_retail.index.name = 'Timestamp'

print("✅ High-Frequency Retail Data Loaded!")
print(f"Shape: {df_retail.shape}")
df_retail.head()


Pandas Version: 3.0.3


ValueError: Invalid frequency: H. Failed to parse with error message: ValueError("Invalid frequency: H. Failed to parse with error message: KeyError('H'). Did you mean h?") Did you mean h?

# 📘 Module 1: Time-Based Indexing & Slicing

## 🧠 1. Theory
When your DataFrame is indexed by a `DatetimeIndex`, Pandas allows you to slice data using **partial string indexing**. You don't need to specify exact start and end timestamps; you can just ask for "March" or "2023".

## 💡 2. Mental Model
A **Smart Calendar**: You don't need to tell it "From March 1st 00:00:00 to March 31st 23:59:59". You just say "Give me March", and it understands.

## 📊 3. Visual Diagram
```text
df.loc['2023']       -> Returns ALL data in 2023
df.loc['2023-02']    -> Returns ALL data in Feb 2023
df.loc['2023-02-14'] -> Returns ALL data on Valentine's Day
```

## 🛠️ 4. Syntax
```python
df.loc['YYYY']
df.loc['YYYY-MM']
df.loc['YYYY-MM-DD']
df.between_time(start_time, end_time) # For intraday slicing
```

## 💻 5. Example & 6. Output
```python
# Partial string indexing
feb_data = df_retail.loc['2023-02']
print(f"February rows: {len(feb_data)}")

# Slicing a specific range
valentines = df_retail.loc['2023-02-14']
print(f"Valentine's Day rows: {len(valentines)}") # 24 hours
```

## 🌍 8. Real World Example
**Business Reporting:** Extracting Q1 data quickly using `df.loc['2023-01':'2023-03']` without needing to calculate the exact millisecond of the quarter's end.

## ⚠️ 9. Common Mistakes
**The Unsorted Index Trap:**
```python
df_retail.loc['2023-02':'2023-01'] # ❌ Returns empty if slicing backwards on unsorted index!
```
**Correct:** Time slicing requires the DatetimeIndex to be monotonically increasing (sorted).

## 🎤 10. Interview Question
**Q:** How do you extract only the data that occurred between 9:00 AM and 5:00 PM across the entire dataset?
**A:** Use `df.between_time('09:00', '17:00')`. This is incredibly useful for isolating "business hours" from 24/7 server logs or retail data.

## 🔮 11. Output Prediction
```python
# What does this return?
print(df_retail.loc['2023-01-01 10:00':'2023-01-01 12:00'])
```
**A:** It returns exactly 3 rows: 10:00, 11:00, and 12:00 on Jan 1st. (Pandas time slicing is **inclusive** on both ends!).

## 🎯 12-16. Practice Tasks
*   **Level 1 (Easy):** Extract all data for the month of January using string slicing.
*   **Level 2 (Medium):** Extract data specifically for the week of Jan 1st to Jan 7th.
*   **Level 3 (Hard):** Use `.between_time()` to filter the dataset to only include "Night Shift" hours (10:00 PM to 06:00 AM). *Hint: Night shift crosses midnight!*
*   **Level 4 (Expert):** Create a boolean mask to filter rows where the date is a Weekend (Saturday/Sunday) AND the time is between 12:00 PM and 8:00 PM.
*   **Level 5 (Challenge):** Write a function `get_quarter_data(df, year, quarter)` that returns the sliced DataFrame using partial string indexing.


<details>
<summary>Click for solution</summary>

```
# ==========================================
# ✅ SOLUTIONS (Module 1)
# ==========================================
# Level 1
jan_data = df_retail.loc['2023-01']
print(f"Jan rows: {len(jan_data)}")

# Level 2
week1 = df_retail.loc['2023-01-01':'2023-01-07']
print(f"Week 1 rows: {len(week1)}")

# Level 3 (Night shift crosses midnight, so we use OR logic or two between_time calls)
# Pandas between_time handles cross-midnight if start > end!
night_shift = df_retail.between_time('22:00', '06:00')
print(f"Night shift rows: {len(night_shift)}")

# Level 4
weekend_mask = df_retail.index.dayofweek >= 5 # 5=Sat, 6=Sun
afternoon_mask = df_retail.index.hour.between(12, 20)
final_filter = df_retail[weekend_mask & afternoon_mask]
print(f"Weekend Afternoon rows: {len(final_filter)}")

# Level 5
def get_quarter_data(df, year, quarter):
    month_map = {1: '03', 2: '06', 3: '09', 4: '12'}
    end_month = month_map[quarter]
    start_month = f"{(int(end_month)-2):02d}"
    return df.loc[f'{year}-{start_month}':f'{year}-{end_month}']

print(get_quarter_data(df_retail, 2023, 1).head())
```

</details>

In [ ]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 1
# ==========================================
# --- YOUR TURN ---
# Level 1: Extract January
# Level 2: Extract Jan 1 to Jan 7
# Level 3: between_time for Night Shift
# Level 4: Weekend + Afternoon mask
# Level 5: get_quarter_data function


# 📘 Module 2: Resampling (Changing Frequency)

## 🧠 1. Theory
**Resampling** means changing the time frequency of your data. 
- **Downsampling:** Going from High Frequency → Low Frequency (e.g., Hourly → Daily). Requires aggregation (`sum`, `mean`).
- **Upsampling:** Going from Low Frequency → High Frequency (e.g., Daily → Hourly). Creates missing data that must be filled (`ffill`, `bfill`, `interpolate`).

## 💡 2. Mental Model
- **Downsampling:** Blurring a high-res photo into a thumbnail (Grouping pixels).
- **Upsampling:** Stretching a small photo (Creating empty pixels that need to be filled).

## 📊 3. Visual Diagram
```text
HOURLY (High Freq)       DAILY (Low Freq)
00:00 | 10               Mon | 240  (Sum of 24 hours)
01:00 | 15       =>      Tue | 310
...                      ...
23:00 | 12
```

## 🛠️ 4. Syntax
```python
# Downsample (Aggregate)
df.resample('D').sum()   # 'D' = Daily, 'W' = Weekly, 'M' = Month End
df.resample('W').mean()

# Upsample (Fill)
df.resample('H').ffill()       # Forward fill
df.resample('H').interpolate() # Linear interpolation
```

## 💻 5. Example & 6. Output
```python
# Downsample: Hourly -> Daily Sum
daily_sales = df_retail.resample('D').sum()
print(daily_sales.head())

# Downsample: Hourly -> Weekly Mean
weekly_avg = df_retail.resample('W').mean()
print(weekly_avg.head())
```

## ⚠️ 9. Common Mistakes
**Forgetting the Aggregation Function:**
```python
df_retail.resample('D') # ❌ Returns a Resampler object, not a DataFrame!
```
**Correct:** You MUST chain an aggregation method like `.sum()`, `.mean()`, or `.apply()`.

## 🎤 10. Interview Question
**Q:** What is the difference between `resample()` and `groupby()`?
**A:** `resample()` is specifically for time-based grouping and uses frequency strings (like `'D'`, `'M'`). `groupby()` is for categorical grouping. Under the hood, `resample()` is actually just a specialized `groupby()` on a `pd.Grouper(freq='D')`.

## 🔮 11. Output Prediction
```python
# What happens to missing hours if we upsample a Daily DataFrame to Hourly?
daily_df = pd.DataFrame({'Sales': [100, 200]}, index=pd.date_range('2023-01-01', periods=2, freq='D'))
print(daily_df.resample('H').sum())
```
**A:** It returns 48 rows. The first 24 rows will have `100` (if using `.sum()` or `.mean()` on the original bucket) or `NaN` depending on the exact alignment, but typically `.sum()` on an upsampled bucket without filling yields `0` or `NaN` for the newly created empty hours. *Correction:* Actually, `resample('H').sum()` on daily data puts the daily sum into the `00:00` hour of that day, and `0` for the rest. To spread it, you'd need `.ffill()` or math.

## 🎯 12-16. Practice Tasks
*   **Level 1 (Easy):** Downsample `df_retail` to **Weekly** totals (`'W'`).
*   **Level 2 (Medium):** Downsample to **Month End** (`'ME'` or `'M'`) calculating both the `sum` of Sales and `mean` of Customers.
*   **Level 3 (Hard):** Upsample the `weekly_avg` DataFrame back to Daily frequency, and fill the missing days using **linear interpolation**.
*   **Level 4 (Expert):** Create a custom resampling rule: Group by "Business Week" (Monday to Friday) and calculate the max daily sales. *Hint: Use `pd.Grouper` or `freq='W'` and filter.*
*   **Level 5 (Challenge):** Write a function that takes a DataFrame and a frequency string, and returns the downsampled data, but ensures the index labels represent the *start* of the period, not the end.


<details>
<summary>Click for solution</summary>

```
# ==========================================
# ✅ SOLUTIONS (Module 2)
# ==========================================
# Level 1
weekly_sales = df_retail.resample('W').sum()
print(weekly_sales.head())

# Level 2 (Note: 'M' is deprecated in Pandas 2.2+, use 'ME' for Month End)
monthly_stats = df_retail.resample('ME').agg({'Sales': 'sum', 'Customers': 'mean'})
print(monthly_stats)

# Level 3
# First create a daily df to upsample
daily_df = df_retail.resample('D').sum()
# Upsample to 12-hourly and interpolate
upsampled = daily_df.resample('12H').interpolate(method='linear')
print(upsampled.head(10))

# Level 4 (Business Week Max)
# W-FRI means weeks ending on Friday. 
biz_week_max = df_retail.resample('W-FRI')['Sales'].max()
print(biz_week_max.head())

# Level 5 (Start of period)
# By default, 'M' labels the end of the month. To get the start, use 'MS' (Month Start)
month_start = df_retail.resample('MS').sum()
print(month_start.head())
```

</details>


In [2]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 2
# ==========================================
# --- YOUR TURN ---
# Level 1: Weekly totals
# Level 2: Month End sum & mean
# Level 3: Upsample & Interpolate
# Level 4: Custom Business Week
# Level 5: Start-of-period labels


# 📘 Module 3: Rolling & Expanding Windows

## 🧠 1. Theory
Instead of grouping data into discrete buckets (like Resampling), **Rolling Windows** perform calculations over a continuous, sliding window of time or rows. This is the foundation of technical analysis (e.g., 7-day Moving Average) and noise reduction.

## 💡 2. Mental Model
A **Sliding Magnifying Glass**: You look at a window of 7 days, calculate the average, then slide the window forward by 1 day, and repeat.

## 📊 3. Visual Diagram
```text
Data:       [10, 20, 30, 40, 50]
Window=3:
Step 1:     [10, 20, 30] -> Mean = 20
Step 2:        [20, 30, 40] -> Mean = 30
Step 3:           [30, 40, 50] -> Mean = 40
Result:     [NaN, NaN, 20, 30, 40]
```

## 🛠️ 4. Syntax
```python
# Time-based window (Highly recommended for Time Series!)
df.rolling(window='3D').mean() 

# Row-based window
df.rolling(window=7).mean()

# Expanding window (Cumulative from the start)
df.expanding().sum()
```

## 💻 5. Example & 6. Output
```python
# 3-Day Rolling Average (Time-based)
# Note: Requires a sorted DatetimeIndex
df_retail['Sales_3D_MA'] = df_retail.rolling(window='3D')['Sales'].mean()

# 7-Day Rolling Standard Deviation (Volatility)
df_retail['Sales_7D_Vol'] = df_retail.rolling(window='7D')['Sales'].std()

print(df_retail[['Sales', 'Sales_3D_MA', 'Sales_7D_Vol']].dropna().head())
```

## ⚠️ 9. Common Mistakes
**The `min_periods` Trap:**
```python
df.rolling(window='7D').mean() 
```
If a 7-day window only has 2 hours of data (because of missing timestamps), it returns `NaN`. 
**Correct:** Use `min_periods=1` if you want it to calculate the mean of whatever data exists within that 7-day span.

## 🎤 10. Interview Question
**Q:** What is the difference between `rolling()` and `expanding()`?
**A:** `rolling()` has a fixed-size window that slides forward (e.g., last 7 days). `expanding()` has a window that starts at the beginning of time and grows with every new row (cumulative sum/mean).

## 🔮 11. Output Prediction
```python
s = pd.Series([1, 2, 3, 4, 5], index=pd.date_range('2023-01-01', periods=5, freq='D'))
print(s.rolling(window='2D').sum())
```
**A:** 
```text
2023-01-01    NaN   (Only 1 day in window, needs 2)
2023-01-02    3.0   (1+2)
2023-01-03    5.0   (2+3)
2023-01-04    7.0   (3+4)
2023-01-05    9.0   (4+5)
```

## 🎯 12-16. Practice Tasks
*   **Level 1 (Easy):** Calculate the 24-hour (1 Day) rolling mean of `Sales`.
*   **Level 2 (Medium):** Calculate the 7-day rolling maximum of `Customers`.
*   **Level 3 (Hard):** Create a "Bollinger Band" style metric: The 7-day rolling mean PLUS 2 times the 7-day rolling standard deviation.
*   **Level 4 (Expert):** Calculate the **Exponential Moving Average (EMA)** using `.ewm(span=7, adjust=False).mean()`.
*   **Level 5 (Challenge):** Write a custom rolling function using `.apply()` that calculates the "Range" (Max - Min) over a 3-day window.


<details>
<summary>Click for solution</summary>

```
# ==========================================
# ✅ SOLUTIONS (Module 3)
# ==========================================
# Level 1
df_retail['Sales_24H_MA'] = df_retail.rolling('24H')['Sales'].mean()

# Level 2
df_retail['Cust_7D_Max'] = df_retail.rolling('7D')['Customers'].max()

# Level 3
roll_mean = df_retail.rolling('7D')['Sales'].mean()
roll_std = df_retail.rolling('7D')['Sales'].std()
df_retail['Upper_Band'] = roll_mean + (2 * roll_std)

# Level 4
df_retail['Sales_EMA'] = df_retail['Sales'].ewm(span=7, adjust=False).mean()

# Level 5
def custom_range(x):
    return x.max() - x.min()

df_retail['Range_3D'] = df_retail.rolling('3D')['Sales'].apply(custom_range, raw=True)
print(df_retail.dropna().head())
```

</details>

In [3]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 3
# ==========================================
# --- YOUR TURN ---
# Level 1: 24H rolling mean
# Level 2: 7D rolling max
# Level 3: Bollinger Upper Band
# Level 4: EWM
# Level 5: Custom rolling apply


# 📘 Module 4: Time Shifts, Lags & Differences
## 🧠 1. Theory
To measure **change**, you need to compare the present to the past. 
- `shift()`: Moves data up/down the index (creates lags/leads).
- `diff()`: Subtracts the previous value from the current value.
- `pct_change()`: Calculates the percentage change (growth rate).

## 💡 2. Mental Model
- **Shift:** Moving boxes on a conveyor belt to align them with yesterday's boxes.
- **Diff:** Checking your bank account: `Today's Balance - Yesterday's Balance = Net Change`.

## 📊 3. Visual Diagram
```text
Sales:      [100, 150, 130]
shift(1):   [NaN, 100, 150]  (Yesterday's Sales)
diff():     [NaN,  50, -20]  (Absolute Change)
pct_change: [NaN, 0.5, -0.13] (Growth Rate %)
```

## 🛠️ 4. Syntax
```python
df['Yesterday_Sales'] = df['Sales'].shift(1)
df['Daily_Change'] = df['Sales'].diff()
df['Growth_Rate'] = df['Sales'].pct_change()

# Shift by time (e.g., exactly 1 week ago)
df['Last_Week_Sales'] = df['Sales'].shift(freq='7D') 
```

## 💻 5. Example & 6. Output
```python
# Let's work with Daily data for cleaner shift examples
daily_df = df_retail.resample('D').sum()

daily_df['Prev_Day_Sales'] = daily_df['Sales'].shift(1)
daily_df['Daily_Diff'] = daily_df['Sales'].diff()
daily_df['Growth_Pct'] = daily_df['Sales'].pct_change()

print(daily_df.head())
```

## ⚠️ 9. Common Mistakes
**Shifting without sorting:**
If your DatetimeIndex has gaps or is unsorted, `shift(1)` shifts by *row position*, not by *time*. 
**Correct:** Always `df.sort_index()` before shifting, or use `shift(freq='1D')` which shifts by actual time, leaving `NaN` if the exact previous day is missing.

## 🎤 10. Interview Question
**Q:** How do you calculate the Year-over-Year (YoY) growth rate for a daily dataset?
**A:** You shift the data by 1 year using time-aware shifting, then calculate percentage change:
`df['Sales'].pct_change(periods=365)` OR `df['Sales'] / df['Sales'].shift(freq='1Y') - 1`.

## 🔮 11. Output Prediction
```python
s = pd.Series([10, 20, 30], index=pd.date_range('2023-01-01', periods=3, freq='D'))
print(s.shift(freq='1D'))
```
**A:**
```text
2023-01-02    10.0
2023-01-03    20.0
2023-01-04    30.0
Freq: D, dtype: float64
```
*Notice the index itself shifted forward by 1 day!*

## 🎯 12-16. Practice Tasks
*   **Level 1 (Easy):** Create a column `Sales_Yesterday` using `shift(1)`.
*   **Level 2 (Medium):** Calculate the absolute difference in `Customers` compared to the same day last week (shift by 7 days).
*   **Level 3 (Hard):** Calculate the 3-day rolling percentage change (Hint: combine `rolling` and a custom function, or use `pct_change` on a resampled dataset).
*   **Level 4 (Expert):** Create a "Streak" counter: How many consecutive days did Sales increase? (Hint: Use `diff() > 0`, then `groupby` on the inverse of `cumsum()`).
*   **Level 5 (Challenge):** Align today's sales with yesterday's weather (mock data) using `merge_asof()` or time-based shifting.


<details>
<summary>Click for solution</summary>

```
# ==========================================
# ✅ SOLUTIONS (Module 4)
# ==========================================
# Level 1
daily_df['Sales_Yesterday'] = daily_df['Sales'].shift(1)

# Level 2 (Shift by 7 days using freq)
daily_df['Cust_Diff_vs_Last_Week'] = daily_df['Customers'] - daily_df['Customers'].shift(freq='7D')

# Level 3 (Rolling Pct Change is tricky, usually done via log returns or custom apply)
# Simple approach: Resample to 3-day blocks, then pct_change
daily_df['Rolling_3D_Pct'] = daily_df['Sales'].rolling('3D').apply(lambda x: (x.iloc[-1] - x.iloc[0]) / x.iloc[0], raw=False)

# Level 4 (Consecutive Increases)
daily_df['Increased'] = daily_df['Sales'].diff() > 0
# Create groups of consecutive Trues
daily_df['Streak_Group'] = (~daily_df['Increased']).cumsum()
# Count within groups
daily_df['Increase_Streak'] = daily_df.groupby('Streak_Group')['Increased'].cumsum()
print(daily_df[['Sales', 'Increased', 'Increase_Streak']].head(10))

# Level 5 (Concept)
# weather_df = pd.DataFrame({'Temp': [20, 22, 21]}, index=pd.date_range('2023-01-01', periods=3, freq='D'))
# pd.merge_asof(daily_df, weather_df, left_index=True, right_index=True, direction='backward')
```

</details>

In [ ]:
# ==========================================
# 💻 CODE & PRACTICE: MODULE 4
# ==========================================
daily_df = df_retail.resample('D').sum()

# --- YOUR TURN ---
# Level 1: Sales_Yesterday
# Level 2: Diff vs Last Week
# Level 3: Rolling pct change
# Level 4: Consecutive increase streak
# Level 5: merge_asof concept


# 🏢 Mini Project: The Executive Retail Dashboard

## 🚀 Capstone Challenge: End-to-End Time Series Pipeline
You are the Lead Data Scientist. The CEO wants a daily morning briefing dashboard.

**Requirements:**
1. **Filter:** Extract only "Business Hours" (09:00 to 18:00).
2. **Resample:** Aggregate to Daily totals.
3. **Feature Engineering:** 
   - Calculate the 7-Day Moving Average of Sales.
   - Calculate the Day-over-Day (DoD) percentage growth.
   - Flag "Anomaly" days where Sales dropped by more than 20% compared to the 7-day average.
4. **Visualize:** Plot the Daily Sales, the 7-Day MA, and highlight the Anomaly days in red.


In [ ]:
# ==========================================
# 🏢 MINI PROJECT SOLUTION
# ==========================================
# 1. Filter Business Hours
biz_hours = df_retail.between_time('09:00', '18:00')

# 2. Resample to Daily
daily_biz = biz_hours.resample('D').sum()

# 3. Feature Engineering
daily_biz['7D_MA'] = daily_biz['Sales'].rolling('7D').mean()
daily_biz['DoD_Growth'] = daily_biz['Sales'].pct_change()

# Flag Anomalies: Sales < 80% of the 7-Day MA
daily_biz['Is_Anomaly'] = daily_biz['Sales'] < (0.8 * daily_biz['7D_MA'])

# 4. Visualize
plt.figure(figsize=(14, 6))

# Plot Daily Sales
plt.plot(daily_biz.index, daily_biz['Sales'], label='Daily Sales', color='skyblue', marker='o', markersize=4)

# Plot 7D Moving Average
plt.plot(daily_biz.index, daily_biz['7D_MA'], label='7-Day MA', color='orange', linewidth=2)

# Highlight Anomalies
anomalies = daily_biz[daily_biz['Is_Anomaly']]
plt.scatter(anomalies.index, anomalies['Sales'], color='red', s=100, label='Anomaly (< -20% vs MA)', zorder=5)

plt.title('Executive Daily Sales Dashboard with Anomaly Detection', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"🚨 Total Anomaly Days Detected: {anomalies.shape[0]}")
print(anomalies[['Sales', '7D_MA', 'DoD_Growth']])


# 🎉 Congratulations!
You have mastered **Pandas Part 8: Advanced Time Series**.
You now know how to:
✅ Slice data intelligently using partial strings and `between_time()`.
✅ Change data frequencies using `resample()` (Downsampling & Upsampling).
✅ Calculate moving averages and volatility using `rolling()` and `expanding()`.
✅ Measure change over time using `shift()`, `diff()`, and `pct_change()`.
✅ Build real-world anomaly detection pipelines.

### 🚀 Next Steps
You have conquered the core Pandas curriculum! 
**What's next?**
- **Part 9:** Performance Optimization (Vectorization, `eval()`, `query()`, PyArrow backend).
- **Part 10:** Connecting to SQL, APIs, and building automated ETL pipelines.
